In [91]:
from Policy.eps_greedy import epsilon_greedy
from Custom_Envs.easy21 import Easy21

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [92]:
def Monte_Carlo_control(space_size,env,policy,num_episodes=1000000, N0=100):
    q_val = np.zeros(space_size)
    counts = np.zeros(space_size)

    for _ in range(num_episodes):
        state_t, terminated = env.reset()
        history = []
        
        while(not terminated):
            eps = N0/(N0+np.sum(counts[state_t[0]-1,state_t[1]-1]))
            action_t = policy(q_val[state_t[0]-1,state_t[1]-1],eps)
            [state_t_1, r_t_1, terminated] = env.step(action_t)

            history.append([state_t,action_t,r_t_1,state_t_1])
            state_t = state_t_1

        cumm_r = 0
        for [s_t,a_t,r,_] in history[::-1]:
            cumm_r += r
            counts[s_t[0]-1,s_t[1]-1,a_t] += 1
            q_val[s_t[0]-1,s_t[1]-1,a_t] += (1/counts[s_t[0]-1,s_t[1]-1,a_t])*(cumm_r-q_val[s_t[0]-1,s_t[1]-1,a_t])

    return(q_val,counts)

In [125]:
Env = Easy21()
space_size = [10,21,2]
optim_q, visit_freq = Monte_Carlo_control(space_size, Env, epsilon_greedy, N0=10000)

In [129]:
def plot_3d_surfaces(x,y,Z,titles=(),heading=""):
    fig = make_subplots(
        rows=1, cols=len(Z),
        subplot_titles=titles,
        specs=[[{'type': 'surface'} for i in range(len(Z))]]
    )

    for idx,z in enumerate(Z):
        fig.add_trace(
            go.Surface(z=z, x=x, y=y, colorscale='Viridis', showscale=False),
            row=1, col=idx+1
        )

    fig.update_layout(title_text=heading, height=600, width=1000)
    fig.update_scenes(
        xaxis_title='Player Sum',
        yaxis_title='Dealer Showing',
        zaxis_title='Value',
    )

    fig.show()

In [130]:
X = np.arange(1,21)
Y = np.arange(1,10)
plot_3d_surfaces(X, Y, [optim_q[:,:,0], optim_q[:,:,1]], titles=["Player Hits","Player Sticks"], heading="MC Optimal Q-Value Fn")

In [131]:
X = np.arange(1,21)
Y = np.arange(1,10)
plot_3d_surfaces(X, Y, [np.max(optim_q, axis=2)], titles=[""], heading="MC Optimal State-Value Fn")

In [132]:
X = np.arange(1,21)
Y = np.arange(1,10)
plot_3d_surfaces(X, Y, [visit_freq[:,:,0], visit_freq[:,:,1]], titles=["Player Hits","Player Sticks"], heading="Visit Freq during Learning")

## N0 = 100

In [133]:
Env = Easy21()
space_size = [10,21,2]
optim_q, visit_freq = Monte_Carlo_control(space_size, Env, epsilon_greedy, num_episodes=1000000, N0=100)

In [134]:
X = np.arange(1,21)
Y = np.arange(1,10)
plot_3d_surfaces(X, Y, [optim_q[:,:,0], optim_q[:,:,1]], titles=["Player Hits","Player Sticks"], heading="MC Optimal Q-Value Fn")

In [135]:
X = np.arange(1,21)
Y = np.arange(1,10)
plot_3d_surfaces(X, Y, [visit_freq[:,:,0], visit_freq[:,:,1]], titles=["Player Hits","Player Sticks"], heading="Visit Freq during Learning")